**Fin 585**  
**Diether**  
**Problem Set**  

**Testing the CAPM using Analyst Disagreement Portfolios**

The primary purpose of this problem set is to give you a portfolio formation task that makes you go through all five steps of our portfolio formation framework including testing the CAPM as a model.

1. Data Preparation.

2. Create portfolio formation or criterion variable.

3. Bin the data based on the formation variable.

4. Portfolio creation using the bins.

5. Test the historical performance and test a model.

A secondary goal is to introduce another interesting portfolio strategy. It produces a large spread in average return. Given that, it's a good set of portfolios for testing the CAPM.

To accomplish the programming tasks, you should be able to adapt a lot of code we've used before, and apply it this situation. <br><br>

**Overview**

In this problem set you reproduce another important empirical result in academic finance. Specifically, you reproduce the **dispersion effect** (or the analyst disagreement effect) of Diether, Malloy, and Scherbina (2002). This empirical result spawned a large literature in academic finance, and certainly some quant funds have traded on this effect.

Dispersion (or analyst disagreement) portfolios are formed based on the standard deviation of analyst eps (earnings per share) forecasts over a given period. Here the standard deviation of analyst eps forecasts is the standard deviation across analysts for a given stock and month (most stocks have between 3 to 13 analysts covering them). Diether, Malloy, and Scherbina don't use raw standard deviation. Instead, they scale the standard deviation of analyst forecasts by the absolute value of the mean forecast. Therefore for a given month ($t$), dispersion for stock $i$ is defined as the following:
\begin{align*}
disp_{it} &= \frac{stdev_{it}}{|mean_{it}|}
\end{align*}
DMS form dispersion portfolios using $disp_{i,t-1}$; in other words, they lag dispersion one month. In this homework you will do the same.

There are three datasets for this problem set. The first is the CRSP data (security prices and returns) during the period from January of 1980 to September of 2024. The second is the analyst earnings per share data from IBES. It also covers the period of January of 1980 to September 2024. The frequency for both datasets is monthly. The stock level identifier in the IBES data is called a CUSIP. Consequently, I also included CUSIPs in the CRSP data. The CUSIP and the calendar month uniquely identify the analyst earnings per share observations.

You can download the CRSP data directly using the following link: [the CRSP data](https://diether.org/prephd/08-mstk_80-24.csv). There is also a link on *Learning Suite*. The data contain the following variables:

|Variable | Description                                              |
|---------|----------------------------------------------------------|
|permno   | stock identifier                                         |
|cusip    | stock identifier also in IBES data                       |
|caldt    | calendar date (the day is not truncated to 1)            |
|ret      | monthly return                                           |
|prc      | stock price (not lagged, contemporaneous with returns)   |   


You can download the IBES data directly using the following link: [the IBES data](https://diether.org/prephd/08-ibes_eps_analyst.csv). There is also a link on *Learning Suite*. The data contain the following variables:

|Variable | Description                                          |
|---------|------------------------------------------------------|
|cusip    | stock identifier also in IBES data                   |
|caldt    | calendar date (the day is not truncated to 1)        |
|meanest  | average analyst forecast for that month/stock        |
|stdev    | standard deviation of forecasts for that month/stock |


Finally, to test the CAPM you are going to need a proxy for the market portfolio and for the riskfree rate. Data from these can be found at [Ken French's Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html). For your convenience I have created a csv file that contains both these variables, and it can be loaded directly into a dataframe from my website (see the code below). The `dataframe` contains the excess return on a proxy for the market portfolio (`exmkt`), a proxy for the riskfree rate (`rf`), and some other portfolios you can ignore. The returns from Ken French's library are in percent: raw returns multiplied by 100 (so make sure after forming your portfolios, you multiply your portfolio returns by 100 so it matches the units of the market return and riskfree rate).<br><br>


**Tasks**

1. Form quintile based equal-weight dispersion portfolios where dispersion is lagged one month. Report summary statistics (including a t-test of whether the average return is statistically different from zero for each portfolio). You should exclude low price stocks from your portfolios (price below $5). 

2. Test the CAPM by running a time series CAPM regression for each of the analyst dispersion portfolios:
$$
r_{pt} - r_{ft} = \alpha_p + \beta_{pM}( r_{Mt} - r_{ft}) + \epsilon_{it}
$$
Consolidate all your regression results into one table using the `Regtable` function in the BYU Finance library: [Regtable Docs](https://fin-library.readthedocs.io/en/latest/regtables.html)

3. Interpret the regression results from question 2). What can you infer? Can you reject that the CAPM holds? Is the market portfolio, the tangency portfolio? Explain your answers.

4. Create a spread portfolio that goes 100% long in portfolio 0 and 100% short in portfolio 4. Test the CAPM using this portfolio. Can you reject the CAPM? Explain your answers.

5. Estimate the security market line using the data available for this homework. Specifically, estimate the following line:
$$
E(r_p) = r_f + \beta_{p}\bigl[E(r_M) - r_f\bigr]
$$
You don't need to plot the estimated line, but report your estimates of $r_f$ and $E(r_M) - r_f$ as a line. So something like:
$$
\overline{r}_p = 4\% + \hat{\beta}_p(6\%)
$$

6. Why is the intercept in a time series CAPM regression called an *average abnormal return*? Explain.

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from finance_byu.summarize import summary
from finance_byu.regtables import Regtable

In [3]:
stk = pd.read_csv('08-mstk_80-24.csv',parse_dates=['caldt'])
ibes = pd.read_csv("08-ibes_eps_analyst.csv",parse_dates=['caldt'])

In [4]:
stk['mdt'] = stk['caldt'].values.astype('datetime64[M]')
stk.head(5)

,permno,caldt,cusip,ret,prc,me,mdt
0,10000,1986-01-31,68391610,NaN,4.37500,16.1000,1986-01-01
1,10000,1986-02-28,68391610,-0.257143,3.25000,11.9600,1986-02-01
2,10000,1986-03-31,68391610,0.365385,4.43750,16.3300,1986-03-01
3,10000,1986-04-30,68391610,-0.098592,4.00000,15.1720,1986-04-01
4,10000,1986-05-30,68391610,-0.222656,3.10938,11.7939,1986-05-01


In [5]:
ibes['mdt'] = ibes['caldt'].values.astype('datetime64[M]')
ibes.head(5)

,cusip,caldt,meanest,stdev,mdt
0,00000000,2010-06-17,1.00,0.01,2010-06-01
1,00000000,2010-07-15,0.98,0.02,2010-07-01
2,00000000,2016-04-14,0.25,0.08,2016-04-01
3,00000000,2016-05-19,0.31,0.01,2016-05-01
4,00000000,2016-06-16,0.31,0.01,2016-06-01


In [6]:
ibes = ibes.drop(columns=['caldt'])
stk = stk.merge(ibes,on=['cusip','mdt'],how='left')
stk

,permno,caldt,cusip,ret,prc,me,mdt,meanest,stdev
0,10000,1986-01-31,68391610,NaN,4.37500,16.1000,1986-01-01,NaN,NaN
1,10000,1986-02-28,68391610,-0.257143,3.25000,11.9600,1986-02-01,NaN,NaN
2,10000,1986-03-31,68391610,0.365385,4.43750,16.3300,1986-03-01,NaN,NaN
3,10000,1986-04-30,68391610,-0.098592,4.00000,15.1720,1986-04-01,NaN,NaN
4,10000,1986-05-30,68391610,-0.222656,3.10938,11.7939,1986-05-01,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2741076,93436,2024-05-31,88160R10,-0.028372,178.08000,567932.0000,2024-05-01,2.50,0.38
2741077,93436,2024-06-28,88160R10,0.111186,197.88000,632155.0000,2024-06-01,2.53,0.39
2741078,93436,2024-07-31,88160R10,0.172781,232.07000,741380.0000,2024-07-01,2.56,0.43
2741079,93436,2024-08-30,88160R10,-0.077391,214.11000,684004.0000,2024-08-01,2.34,0.29


**Create the Dispersion Variable and Lagged Variables**

In [7]:
stk['disp'] = stk['stdev'] / stk['meanest'].abs()

stk['displag'] = stk.groupby('permno')['disp'].shift()
stk['prclag'] = stk.groupby('permno')['prc'].shift()

stk

,permno,caldt,cusip,ret,prc,me,mdt,meanest,stdev,disp,displag,prclag
0,10000,1986-01-31,68391610,NaN,4.37500,16.1000,1986-01-01,NaN,NaN,NaN,NaN,NaN
1,10000,1986-02-28,68391610,-0.257143,3.25000,11.9600,1986-02-01,NaN,NaN,NaN,NaN,4.3750
2,10000,1986-03-31,68391610,0.365385,4.43750,16.3300,1986-03-01,NaN,NaN,NaN,NaN,3.2500
3,10000,1986-04-30,68391610,-0.098592,4.00000,15.1720,1986-04-01,NaN,NaN,NaN,NaN,4.4375
4,10000,1986-05-30,68391610,-0.222656,3.10938,11.7939,1986-05-01,NaN,NaN,NaN,NaN,4.0000
...,...,...,...,...,...,...,...,...,...,...,...,...
2741076,93436,2024-05-31,88160R10,-0.028372,178.08000,567932.0000,2024-05-01,2.50,0.38,0.152000,0.248120,183.2800
2741077,93436,2024-06-28,88160R10,0.111186,197.88000,632155.0000,2024-06-01,2.53,0.39,0.154150,0.152000,178.0800
2741078,93436,2024-07-31,88160R10,0.172781,232.07000,741380.0000,2024-07-01,2.56,0.43,0.167969,0.154150,197.8800
2741079,93436,2024-08-30,88160R10,-0.077391,214.11000,684004.0000,2024-08-01,2.34,0.29,0.123932,0.167969,232.0700


<br>

**Task 1**

+ Form equal-weight portfolios based on lagged dispersion.<br><br>

+ Report summary statistics.<br>

+ Going to make a copy of the data and work off of that.

+ Not necessary, but I want to examine a different portfolio formation variable later. $\leftarrow$ extra, not part of the assignment.

In [8]:
df = stk.query("displag == displag and prclag >= 5").reset_index(drop=True)
df['bins'] = df.groupby('caldt')['displag'].transform(pd.qcut,5,labels=False)

ew = (df.groupby(['caldt','bins'])['ret'].mean().unstack(level='bins')
      .rename('p{:.0f}'.format,axis='columns')*100)
ew

bins,p0,p1,p2,p3,p4
caldt,,,,,
1980-01-31,3.458840,4.520233,6.322821,7.072740,7.795961
1980-02-29,-4.657756,-3.846906,-3.544719,-1.588982,-2.649652
1980-03-31,-11.625155,-11.103266,-12.025283,-14.381795,-17.904328
1980-04-30,7.441074,5.905371,5.990816,4.685752,6.111547
1980-05-30,7.610056,6.931914,8.427571,8.556074,8.566147
...,...,...,...,...,...
2024-05-31,2.927383,4.381806,4.159601,2.628200,4.851757
2024-06-28,-1.028287,-0.602192,-1.832743,-2.615258,-2.800799
2024-07-31,7.304960,8.827467,10.512328,8.420995,7.950004


In [9]:
summary(ew).loc[['count','mean','std','tstat','pval'],].round(3)

bins,p0,p1,p2,p3,p4
count,537.000,537.000,537.000,537.000,537.000
mean,1.300,1.206,1.132,1.021,0.687
std,4.689,5.069,5.507,6.113,7.005
tstat,6.423,5.514,4.761,3.871,2.271
pval,0.000,0.000,0.000,0.000,0.024


<br>

**Task 2**

Test the CAPM by running a time series CAPM regression for each of the analyst dispersion portfolios:
$$
r_{pt} - r_{ft} = \alpha_p + \beta_{pM}( r_{Mt} - r_{ft}) + \epsilon_{it}
$$

In [10]:
fac = pd.read_csv('https://diether.org/prephd/08-factors.csv',
                  parse_dates=['caldt'])
fac

,caldt,exmkt,smb,hml,umd,rf
0,1927-01-31,-0.06,-0.37,4.54,0.36,0.25
1,1927-02-28,4.18,0.04,2.94,-2.14,0.26
2,1927-03-31,0.13,-1.65,-2.61,3.61,0.30
3,1927-04-30,0.46,0.30,0.81,4.30,0.25
4,1927-05-31,5.44,1.53,4.73,3.00,0.30
...,...,...,...,...,...,...
1168,2024-05-31,4.34,0.78,-1.67,-0.02,0.44
1169,2024-06-28,2.77,-3.06,-3.31,0.90,0.41
1170,2024-07-31,1.24,6.80,5.74,-2.42,0.45
1171,2024-08-30,1.61,-3.55,-1.13,4.79,0.48


In [11]:
ew = ew.join(fac.set_index('caldt'),how='inner')
ew

,p0,p1,p2,p3,p4,exmkt,smb,hml,umd,rf
caldt,,,,,,,,,,
1980-01-31,3.458840,4.520233,6.322821,7.072740,7.795961,5.51,1.62,1.75,7.55,0.80
1980-02-29,-4.657756,-3.846906,-3.544719,-1.588982,-2.649652,-1.22,-1.85,0.61,7.88,0.89
1980-03-31,-11.625155,-11.103266,-12.025283,-14.381795,-17.904328,-12.90,-6.64,-1.01,-9.55,1.21
1980-04-30,7.441074,5.905371,5.990816,4.685752,6.111547,3.97,1.05,1.06,-0.43,1.26
1980-05-30,7.610056,6.931914,8.427571,8.556074,8.566147,5.26,2.13,0.38,-1.12,0.81
...,...,...,...,...,...,...,...,...,...,...
2024-05-31,2.927383,4.381806,4.159601,2.628200,4.851757,4.34,0.78,-1.67,-0.02,0.44
2024-06-28,-1.028287,-0.602192,-1.832743,-2.615258,-2.800799,2.77,-3.06,-3.31,0.90,0.41
2024-07-31,7.304960,8.827467,10.512328,8.420995,7.950004,1.24,6.80,5.74,-2.42,0.45


In [12]:
names = ['exp0','exp1','exp2','exp3','exp4']
ew[names] = ew[[s[2:] for s in names]].sub(ew['rf'],axis='index')
ew

,p0,p1,p2,p3,p4,exmkt,smb,hml,umd,rf,exp0,exp1,exp2,exp3,exp4
caldt,,,,,,,,,,,,,,,
1980-01-31,3.458840,4.520233,6.322821,7.072740,7.795961,5.51,1.62,1.75,7.55,0.80,2.658840,3.720233,5.522821,6.272740,6.995961
1980-02-29,-4.657756,-3.846906,-3.544719,-1.588982,-2.649652,-1.22,-1.85,0.61,7.88,0.89,-5.547756,-4.736906,-4.434719,-2.478982,-3.539652
1980-03-31,-11.625155,-11.103266,-12.025283,-14.381795,-17.904328,-12.90,-6.64,-1.01,-9.55,1.21,-12.835155,-12.313266,-13.235283,-15.591795,-19.114328
1980-04-30,7.441074,5.905371,5.990816,4.685752,6.111547,3.97,1.05,1.06,-0.43,1.26,6.181074,4.645371,4.730816,3.425752,4.851547
1980-05-30,7.610056,6.931914,8.427571,8.556074,8.566147,5.26,2.13,0.38,-1.12,0.81,6.800056,6.121914,7.617571,7.746074,7.756147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-05-31,2.927383,4.381806,4.159601,2.628200,4.851757,4.34,0.78,-1.67,-0.02,0.44,2.487383,3.941806,3.719601,2.188200,4.411757
2024-06-28,-1.028287,-0.602192,-1.832743,-2.615258,-2.800799,2.77,-3.06,-3.31,0.90,0.41,-1.438287,-1.012192,-2.242743,-3.025258,-3.210799
2024-07-31,7.304960,8.827467,10.512328,8.420995,7.950004,1.24,6.80,5.74,-2.42,0.45,6.854960,8.377467,10.062328,7.970995,7.500004


In [13]:
reg0 = smf.ols('exp0 ~ 1 + exmkt',data=ew).fit()
reg0.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   exp0   R-squared:                       0.842
Model:                            OLS   Adj. R-squared:                  0.842
Method:                 Least Squares   F-statistic:                     2847.
Date:                Wed, 04 Mar 2026   Prob (F-statistic):          2.19e-216
Time:                        14:37:43   Log-Likelihood:                -1096.4
No. Observations:                 537   AIC:                             2197.
Df Residuals:                     535   BIC:                             2205.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.2849      0.082      3.491      0.001       0.125       0.445
exmkt          0.9512      0.018     53.361      0.000       0.916       0.986
==============================================================================
Omnibus:                       14.294   Durbin-Watson:                   1.712
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               27.191
Skew:                           0.094   Prob(JB):                     1.25e-06
Kurtosis:                       4.086   Cond. No.                         4.64
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [14]:
reg = [smf.ols(f'{y} ~ exmkt',data=ew).fit() for y in names]
reg

In [15]:
Regtable(reg,stat='tstat',sig='coeff').render()

,exp0,exp1,exp2,exp3,exp4
Intercept,0.285***,0.126,-0.007,-0.191,-0.623***
,(3.49),(1.53),(-0.07),(-1.68),(-4.22)
exmkt,0.951***,1.041***,1.122***,1.225***,1.361***
,(53.36),(57.74),(54.37),(49.19),(42.18)
obs,537,537,537,537,537
Rsq,0.84,0.86,0.85,0.82,0.77


In [16]:
Regtable(reg,stat='pvalues',sig='coeff').render()

,exp0,exp1,exp2,exp3,exp4
Intercept,0.285***,0.126,-0.007,-0.191,-0.623***
,(0.00),(0.13),(0.95),(0.09),(0.00)
exmkt,0.951***,1.041***,1.122***,1.225***,1.361***
,(0.00),(0.00),(0.00),(0.00),(0.00)
obs,537,537,537,537,537
Rsq,0.84,0.86,0.85,0.82,0.77


<br>

**Task 3**

+ Interpret the regression results from question 2). What can you infer? Can you reject that the CAPM holds? Is the market portfolio, the tangency portfolio? Explain your answers.

+ Let's talk about it.<br><br>


**Extra: GRS Test**

+ We're doing five tests of the CAPM at the same time. We should take that into account.

+ Can do a joint F-test under the null that all the alpha = 0. $\leftarrow$ joint F-test.

+ Called a GRS test in finance; GRS stands for Gibbons, Ross, and Shanken.

+ There is a GRS test function in the BYU Finance library: [GRS Docs](https://fin-library.readthedocs.io/en/latest/statistics.html#statistics)

In [17]:
from finance_byu.statistics import GRS

grsstat,pval,tbl = GRS(ew,names,['exmkt'])
print(f'GRS = {grsstat:.2f} and p-value = {pval:.5g}\n')

GRS = 10.99 and p-value = 4.396e-10



<br>

**Task 4**

Create a spread portfolio that goes 100% long in portfolio 0 (p0) and 100% short in portfolio 4 (p4). Test the CAPM using this portfolio. Can you reject the CAPM? Explain your answers.

In [18]:
ew['spread'] = ew['p0'] - ew['p4']
summary(ew[names + ['spread']]).loc[['count','mean','std','tstat'],].round(3)

,exp0,exp1,exp2,exp3,exp4,spread
count,537.000,537.000,537.000,537.000,537.000,537.000
mean,0.971,0.877,0.802,0.692,0.357,0.613
std,4.692,5.076,5.519,6.127,7.022,3.657
tstat,4.794,4.004,3.369,2.618,1.180,3.885


In [19]:
reg = [smf.ols(f'{y} ~ exmkt',data=ew).fit() for y in names + ['spread']]
Regtable(reg,stat='tstat',sig='coeff').render()

,exp0,exp1,exp2,exp3,exp4,spread
Intercept,0.285***,0.126,-0.007,-0.191,-0.623***,0.908***
,(3.49),(1.53),(-0.07),(-1.68),(-4.22),(6.59)
exmkt,0.951***,1.041***,1.122***,1.225***,1.361***,-0.409***
,(53.36),(57.74),(54.37),(49.19),(42.18),(-13.59)
obs,537,537,537,537,537,537
Rsq,0.84,0.86,0.85,0.82,0.77,0.26


<br>

**Task 5**

+ Estimate the security market line using the data available for this homework. Specifically, estimate the following line:
$$
E(r_p) = r_f + \beta_{p}\bigl[E(r_M) - r_f\bigr]
$$

+ Answer just plug in the sample moments for $r_f$ and $E(r_M) - r_f$ $\rightarrow$
$$
\overline{r}_p = 0.296\% + \hat{\beta}_p(0.755\%)
$$

In [20]:
summary(ew[['exmkt','rf']]).loc[['count','mean','std','tstat'],].round(3)

,exmkt,rf
count,537.000,537.000
mean,0.721,0.329
std,4.525,0.287
tstat,3.691,26.542


<br>

**Task 6**

+ Why is the intercept in a time series CAPM regression called an *average abnormal return*? Explain.

+ Let's talk about it.

**Extra: Analyst Disagreement Lagging Three Months**

+ Instead of disp lagged one month, lets lag in three months.

+ This should help us understand how timely the dispersion signal needs to be.

+ If the effect is still strong using a three month lag, then maybe we could go to quarterly rebalancing if we implemented this strategy.

In [21]:
stk['displag3'] = stk.groupby('permno')['disp'].shift(3)
df = stk.query("displag3 == displag3 and prclag >= 5").reset_index(drop=True)

df['bins3'] = df.groupby('caldt')['displag3'].transform(pd.qcut,5,labels=False)

ew = (df.groupby(['caldt','bins3'])['ret'].mean().unstack(level='bins3')
      .rename('p{:.0f}'.format,axis='columns')*100 )

ew['spread'] = ew['p0'] - ew['p4']
summary(ew).loc[['count','mean','std','tstat','pval'],].round(3)

bins3,p0,p1,p2,p3,p4,spread
count,535.000,535.000,535.000,535.000,535.000,535.000
mean,1.249,1.188,1.102,1.008,0.743,0.506
std,4.705,5.016,5.496,6.025,6.968,3.587
tstat,6.137,5.480,4.637,3.869,2.466,3.260
pval,0.000,0.000,0.000,0.000,0.014,0.001


In [22]:
fac = pd.read_csv('https://diether.org/prephd/08-factors.csv',
                  parse_dates=['caldt'])

ew = ew.join(fac.set_index('caldt'),how='inner')

names = ['exp0','exp1','exp2','exp3','exp4']
ew[names] = ew[[s[2:] for s in names]].sub(ew['rf'],axis='index')

reg = [smf.ols(f'{y} ~ exmkt',data=ew).fit() for y in names]
reg

reg = [smf.ols(f'{y} ~ exmkt',data=ew).fit() for y in names + ['spread']]
Regtable(reg,stat='tstat',sig='coeff').render()

,exp0,exp1,exp2,exp3,exp4,spread
Intercept,0.238***,0.125,-0.028,-0.188,-0.550***,0.787***
,(2.91),(1.52),(-0.30),(-1.72),(-3.71),(5.78)
exmkt,0.956***,1.029***,1.122***,1.215***,1.349***,-0.394***
,(53.68),(57.40),(55.07),(50.92),(41.70),(-13.23)
obs,535,535,535,535,535,535
Rsq,0.84,0.86,0.85,0.83,0.77,0.25
